##  Assignement 1

## 1. Import the needed libraries

In [1]:
import sys #needed for vscode to recognize the topologicpy package
from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.Wire import Wire
from topologicpy.Face import Face
from topologicpy.Shell import Shell
from topologicpy.Cell import Cell
from topologicpy.CellComplex import CellComplex
from topologicpy.Cluster import Cluster
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Helper import Helper
from topologicpy.Grid import Grid
from topologicpy.Graph import Graph
from topologicpy.Color import Color

## 2. Check the TopologicPy Version

In [2]:
print("This tutorial requires topologicpy version 0.9.18 or newer.")
print(Helper.Version())

This tutorial requires topologicpy version 0.9.18 or newer.
The version that you are using (0.9.21) is OLDER than the latest version (0.9.33) from PyPI. Please consider upgrading to the latest version.


## 3. Set your renderer:
* Visual studio code: "vscode"
* Google Colab: "colab"
* Browser: "browser"

In [3]:
renderer = "vscode"

## 4. Import the OBJ files, Check them and Show them

In [ ]:
# Import all OBJ files
buildings = Topology.ByOBJPath(r"C:\Users\Win11\GraphML_RaniaChihaoui\assets\obj\buildings.obj")


# Check all OBJ files

print("buildings", buildings)



#build the clusters and analyze them
building = Cluster.ByTopologies(buildings)


print("building", building)


# Display the building, doors and windows

Topology.Show(building, Doors, Windows, renderer="vscode")



## 5. Manipulate Geometry Visualization

In [ ]:
# Displaying the grouped geometry
Topology.Show(
    building,
    faceColor=[210, 210, 250],
    faceOpacity=0.5,
    edgeColor="black",
    edgeWidth=3,
    showVertices=False,
    backgroundColor="white",
    width=800,
    height=600,
    renderer="vscode"

)

## Assignement 2

In [ ]:
def reset_dictionaries(shell):
    faces = Topology.Faces(shell)
    for i, f in enumerate(faces):
        d = Topology.Dictionary(f)
        keys = Dictionary.Keys(d)
        for key in keys:
            if not key == "face_id":
                d = Dictionary.RemoveKey(d, key)
        f = Topology.SetDictionary(f, d)

def transfer_dicts_by_key(topologies, selectors, key):
    dicts = {}
    for t in topologies:
        d = Topology.Dictionary(t)
        value = Dictionary.ValueAtKey(d, key, None)
        if value:
            dicts[str(value)] = t
    
    for s in selectors:
        d = Topology.Dictionary(s)
        value = Dictionary.ValueAtKey(d, key, None)
        if value:
            f = dicts[str(value)]
            f = Topology.SetDictionary(f, d)


In [ ]:
Topology.Show(cc_building,
              camera=[0,0,6],
              faceColor=[210,210,250],
              faceOpacity=1,
              edgeColor="white",
              edgeWidth=3,
              showVertices=False,
              backgroundColor="black",
              width=800,
              height=600,
              renderer = renderer)

In [ ]:
b_r = Wire.BoundingRectangle(cc_building)
d = Topology.Dictionary(b_r)
xmin = Dictionary.ValueAtKey(d, "xmin")
xmax = Dictionary.ValueAtKey(d, "xmax")
ymin = Dictionary.ValueAtKey(d, "ymin")
ymax = Dictionary.ValueAtKey(d, "ymax")
width = Dictionary.ValueAtKey(d, "width")
length = Dictionary.ValueAtKey(d, "length")
uRange = list(range(0,int(width)+3,3))
vRange = list(range(0,int(length)+3,3))

grid = Grid.EdgesByDistances(cc_building, clip=True, uRange=uRange, vRange=vRange)

In [ ]:
Topology.Show(cc_building, grid,
              camera=[0,0,6],
              faceColor=[210,210,250],
              faceOpacity=1,
              edgeColor="grey",
              edgeWidth=3,
              showVertices=False,
              backgroundColor="black",
              width=800,
              height=600,
              renderer = renderer)

In [ ]:
shell = Topology.Slice(cc_building, grid)
faces = Topology.Faces(shell)
# Assign a sequential unique face id to reference it later (e.g. "face_21")
for i, f in enumerate(faces):
    d = Dictionary.ByKeyValue("face_id", "face_"+str(i+1))
    f = Topology.SetDictionary(f, d)

In [ ]:
Topology.Show(shell,
              camera=[0,0,6],
              faceColor=[210,210,250],
              faceOpacity=0.9,
              edgeColor="black",
              edgeWidth=3,
              showVertices=False,
              backgroundColor="black",
              width=800,
              height=600,
              renderer = renderer)

In [ ]:
# Note: Graph nodes automatically inherit the dictionaries of the entities they 
navigation_graph = Graph.ByTopology(shell, direct=False, viaSharedTopologies=True)
analysis_graph = Graph.ByTopology(shell)

In [ ]:
g_verts = Graph.Vertices(analysis_graph)

In [ ]:
Topology.Show(analysis_graph,
              camera=[0,0,6],
              vertexSize=4,
              vertexColor="red",
              edgeColor="lightgrey",
              backgroundColor="black",
              width=800,
              height=600,
              renderer=renderer)
              

In [ ]:
import time

start_vertex = Vertex.ByCoordinates(xmin+2, ymax-2,0) # Upper left corner
end_vertex = Vertex.ByCoordinates(xmax-2,ymin+2,0) # Lower right corner
crg = Graph.CompiledRoutingGraph(navigation_graph, precomputeTurns=False)
start = time.time()
shortest_path = Graph.ShortestPath(crg, vertexA=start_vertex, vertexB=end_vertex)
end = time.time()
print("Shortest Path Duration:", round(end-start, 2), "seconds")

# Straighten the shortest path (optional)
start = time.time()
straight_path = Wire.Straighten(shortest_path, host=cc_building)
end = time.time()
print("Straighten Wire Duration:", round(end-start, 2), "seconds")

print("Original Shortest Path Length:", round(Wire.Length(shortest_path), 2))
print("Straightened Shortened Path Length:", round(Wire.Length(straight_path), 2))
edges = Topology.Edges(shortest_path)
for edge in edges:
    d = Dictionary.ByKeysValues(["width", "color"], [7, "red"])
    edge = Topology.SetDictionary(edge, d)
edges = Topology.Edges(straight_path)
for edge in edges:
    d = Dictionary.ByKeysValues(["width", "color"], [7, "blue"])
    edge = Topology.SetDictionary(edge, d)

    Topology.Show(cc_building, shortest_path, straight_path,
              camera=[0,0,6],
              faceColor=[210,210,250],
              faceOpacity=1,
              edgeColorKey="color",
              edgeWidthKey="width",
              backgroundColor="black",
              width=800,
              height=600,
              renderer=renderer)

Shortest Path Duration: 0.0 seconds
Straighten Wire Duration: 0.09 seconds
Original Shortest Path Length: 8.54
Straightened Shortened Path Length: 7.5
